# Soup: evidence-first LoRA experiment

**Qwen3-1.7B + xLAM; official environment: free Colab T4.**

Select **Runtime → Change runtime type → T4 GPU**. Run cells in order. Source is embedded from the locally tested project. No token is embedded. Access and reference review are explicit checkpoints. All model results below are produced by the runtime; this distributed notebook has not yet been executed.

The task is offline call prediction for an automatic-execution verdict. No external tools are executed. Offline benchmark success does not establish deployment safety.

In [ ]:
from pathlib import Path
import base64, io, os, zipfile, sys, subprocess
project = Path('/content/soup_submission')
project.mkdir(exist_ok=True)
source_payload = """
...
"""
with zipfile.ZipFile(io.BytesIO(base64.b64decode(source_payload))) as z:
    z.extractall(project)
os.chdir(project)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Source installed. Colab CUDA torch was not replaced with a CPU wheel.')

## Hardware and local logic checks

This checks the actual hardware and runs synthetic scorer/model tests. Synthetic tests are implementation evidence, not xLAM benchmark results.

In [ ]:
from soup_task.runtime import require_t4, environment
from soup_task.core import utc
import json
require_t4()
print(json.dumps(environment(), indent=2))
test = subprocess.run([sys.executable, '-m', 'pytest', '-q'], capture_output=True, text=True)
print(test.stdout, test.stderr)
(project/'LOCAL_TESTS.txt').write_text(utc()+'\n'+test.stdout+test.stderr)
test.check_returncode()

## Authorized dataset access

Accept the conditions on the [xLAM dataset page](https://huggingface.co/datasets/Salesforce/xlam-function-calling-60k) in your Hugging Face account. Upload an already authorized source JSON (or .json.gz) to /content/, or enter a read token in the private prompt. Do not put credentials in a code cell or saved output.

In [ ]:
from getpass import getpass
import gzip, shutil
local_dataset = Path('/content/xlam_function_calling_60k.json')
compressed_dataset = local_dataset.with_suffix('.json.gz')
if compressed_dataset.is_file() and not local_dataset.exists():
    with gzip.open(compressed_dataset, 'rb') as source, local_dataset.open('wb') as target:
        shutil.copyfileobj(source, target)
if local_dataset.is_file():
    os.environ['XLAM_LOCAL_FILE'] = str(local_dataset)
elif not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass('Hugging Face read token (hidden): ')
from datetime import datetime, timezone
run = project/'runs'/('colab-'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
subprocess.run([sys.executable, '-u', 'experiment.py', 'prepare', '--run-dir', str(run)], check=True)
(run/'evidence/local_tests.txt').write_text((project/'LOCAL_TESTS.txt').read_text())
print('Prepared:', run)

## Review references and preserve the pre-training estimate

Read the 40 references: check required values, tool choice, ambiguity, and labels. Do not approve an unaudited sample. If references need exclusion, preserve this attempt and update the selection rules before creating a fresh run. The estimate is immutable.

In [ ]:
from IPython.display import display, Markdown
display(Markdown((run/'data/AUDIT.md').read_text()))
print((run/'evidence/memory_estimate_before_training.json').read_text())
reviewer = input('Who reviewed these references? State AI assistance if applicable: ').strip()
notes = input('Record findings and limitations; enter nothing to stop: ').strip()
if not reviewer or not notes:
    raise RuntimeError('Reference review was not recorded.')
subprocess.run([sys.executable, 'experiment.py', 'approve-audit', '--run-dir', str(run),
                '--reviewer', reviewer, '--notes', notes], check=True)

## Run the measured pipeline

The stages are pre-flight, LoRA + zero-LR control, GPU FP32 merge, NF4 export, fresh artifact comparisons, Soup diagnostics, verification, and report packaging. Each stage gets timestamped logs, allocator/RSS measurements, and raw nvidia-smi output. A failure stops execution and preserves the attempt. Reconnecting does not guarantee recovery if Colab deletes the VM; download evidence/checkpoints when available.

In [ ]:
subprocess.run([sys.executable, '-u', 'experiment.py', 'run', '--run-dir', str(run)], check=True)

## Inspect and download evidence

A negative verdict is an honest result. Read the reasons and raw outputs. Review the two-page PDF visually, the reflection draft, and AI-use notes before submission. Save this notebook with outputs after clearing any sensitive accidental input.

In [ ]:
from google.colab import files
print((run/'results/verification.json').read_text())
files.download(str(run/'submission/report.pdf'))
files.download(str(run/'submission/supporting_files.zip'))
display(Markdown((run/'submission/REFLECTION.md').read_text()))

## Preserve large artifacts separately

The email archive contains the adapter, source, data selections, results, and raw logs, not the multi-gigabyte full weights. Download the merged and NF4 artifacts before releasing the runtime. Use the cell below on one file at a time to avoid concurrent large browser blobs. Download all files in each model directory; keep the shard index and tokenizer files. Compare each downloaded file against `artifact_manifest.json`.

In [ ]:
for kind in ['merged', 'quantized']:
    for path in sorted((run/'artifacts'/kind).glob('*')):
        if path.is_file(): print(path, path.stat().st_size, 'bytes')
# Set one path from the printed list, then run files.download(str(path)).
# Keep the runtime until all required artifacts and evidence are backed up.

## Independent verification

`python verify.py --run-dir <run>` reloads models on the T4 and reruns comparisons in a new verification directory. `--recorded` replays saved predictions/probes and checks artifact hashes; it must not be described as a fresh model evaluation.

Submit by replying to **team@trysoup.dev** in the original thread. The notebook never sends email.